# E01 - SO SÁNH CÁCH GÁN NHÃN CẢM XÚC: 3 LỚP vs 4 LỚP (THÊM MIXED)

**Câu hỏi:** lớp Trung lập kém (F1 ~0.50) là do model hay do nhãn bị trộn câu khen-chê lẫn lộn?

**Cố định:** cùng TF-IDF + LogisticRegression, chỉ đổi cách gán nhãn. Chỉ đánh giá trên **dev**, chưa đụng test.

In [1]:
import os, sys, re
import pandas as pd
import numpy as np
ROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'experiments' else os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
PROC_DIR = os.path.join(ROOT, 'data', 'processed')
from src.exp_log import log_experiment

In [2]:
df_train = pd.read_csv(os.path.join(PROC_DIR, 'train_clean.csv'))
df_dev = pd.read_csv(os.path.join(PROC_DIR, 'dev_clean.csv'))
df_train.shape, df_dev.shape

((7776, 16), (1110, 16))

## 1. TẠO NHÃN 4 LỚP: câu có cả khía cạnh Positive và Negative -> Mixed (3)

In [3]:
_POL_RE = re.compile(r'#(Positive|Neutral|Negative)\}')
def to_4class(row):
    pols = _POL_RE.findall(str(row['label']))
    if 'Positive' in pols and 'Negative' in pols:
        return 3
    return row['sentiment']
for df in [df_train, df_dev]:
    df['sentiment_4'] = df.apply(to_4class, axis=1)
pd.crosstab(df_train['sentiment'], df_train['sentiment_4'])

sentiment_4,0,1,2,3
sentiment,,,,
0,2129,0,0,388
1,0,747,0,777
2,0,0,2943,792


## 2. CÙNG PIPELINE, HAI CÁCH GÁN NHÃN

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, classification_report
PARAMS = dict(ngram_range=(1, 2), min_df=2, max_df=0.95, max_features=15000, C=1.0, class_weight='balanced')
def make_pipe():
    return Pipeline([
        ('tfidf', TfidfVectorizer(token_pattern=r'(?u)\b\w+\b|\[\w+\]', ngram_range=PARAMS['ngram_range'],
                                  min_df=PARAMS['min_df'], max_df=PARAMS['max_df'],
                                  max_features=PARAMS['max_features'], sublinear_tf=True)),
        ('clf', LogisticRegression(C=PARAMS['C'], class_weight=PARAMS['class_weight'], max_iter=1000, random_state=42)),
    ])
X_tr, X_dev = df_train['sentence_seg'].fillna(''), df_dev['sentence_seg'].fillna('')

In [5]:
results, neu_f1 = {}, {}
for scheme, col, names in [('3class', 'sentiment', ['Neg', 'Neu', 'Pos']),
                           ('4class', 'sentiment_4', ['Neg', 'Neu', 'Pos', 'Mixed'])]:
    pred = make_pipe().fit(X_tr, df_train[col]).predict(X_dev)
    results[scheme] = f1_score(df_dev[col], pred, average='macro')
    neu_f1[scheme] = f1_score(df_dev[col], pred, labels=[1], average='macro')
    print(f'===== {scheme} | dev macro F1 = {results[scheme]:.4f} | Neutral F1 = {neu_f1[scheme]:.4f}')
    print(classification_report(df_dev[col], pred, target_names=names, digits=4))

===== 3class | dev macro F1 = 0.7478 | Neutral F1 = 0.5466
              precision    recall  f1-score   support

         Neg     0.8237    0.8543    0.8387       350
         Neu     0.5039    0.5972    0.5466       216
         Pos     0.9043    0.8162    0.8580       544

    accuracy                         0.7856      1110
   macro avg     0.7440    0.7559    0.7478      1110
weighted avg     0.8010    0.7856    0.7913      1110



===== 4class | dev macro F1 = 0.6826 | Neutral F1 = 0.3667
              precision    recall  f1-score   support

         Neg     0.8061    0.8693    0.8365       306
         Neu     0.3333    0.4074    0.3667       108
         Pos     0.8815    0.8283    0.8541       431
       Mixed     0.7037    0.6453    0.6732       265

    accuracy                         0.7550      1110
   macro avg     0.6811    0.6876    0.6826      1110
weighted avg     0.7649    0.7550    0.7586      1110



## 3. GHI LOG

Lưu ý: macro F1 của 3 lớp và 4 lớp **không so trực tiếp được** vì số lớp khác nhau. Nên nhìn F1 riêng của lớp Neutral ở hai cách.

In [6]:
for scheme, f1 in results.items():
    log_experiment('E01', scheme, 'Label scheme: 3-class vs 4-class (Mixed)', PARAMS, dev_f1=f1,
                   notes=f'TF-IDF+LR, eval dev, neutral_f1={neu_f1[scheme]:.4f}')
pd.read_csv(os.path.join(ROOT, 'reports', 'experiments_log.csv')).tail()

,exp_id,run_name,timestamp,description,params,dev_f1,test_f1,notes
0,E01,3class,2026-11-17 23:21,Label scheme: 3-class vs 4-class (Mixed),"{""ngram_range"": [1, 2], ""min_df"": 2, ""max_df"":...",0.7478,NaN,"TF-IDF+LR, eval dev, neutral_f1=0.5466"
1,E01,4class,2026-11-17 23:21,Label scheme: 3-class vs 4-class (Mixed),"{""ngram_range"": [1, 2], ""min_df"": 2, ""max_df"":...",0.6826,NaN,"TF-IDF+LR, eval dev, neutral_f1=0.3667"


## 4. KẾT LUẬN

_(Điền sau khi chạy: F1 lớp Neutral thay đổi thế nào? Lớp Mixed có học được không? Có nên đổi nhãn trong pipeline chính không?)_